<a href="https://colab.research.google.com/github/pranathi139/GenAI/blob/main/Week2_Hybrid_Retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-openai langchain-community faiss-cpu rank-bm25


In [12]:

import os

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from rank_bm25 import BM25Okapi


In [13]:
docs = [
    Document(page_content="Hybrid retrieval combines vector similarity and keyword matching."),
    Document(page_content="Vector search uses embeddings to retrieve semantically similar text."),
    Document(page_content="Keyword search relies on exact word matching like BM25."),
    Document(page_content="Hybrid retrieval improves factual accuracy in RAG systems."),
    Document(page_content="FAISS is a library for efficient vector similarity search.")
]

In [14]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(docs, embeddings)



/tmp/ipykernel_11620/337680702.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
tokenized_docs = [doc.page_content.split() for doc in docs]
bm25 = BM25Okapi(tokenized_docs)

def bm25_retrieve(query, k=3):
    scores = bm25.get_scores(query.split())
    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    return ranked[:k]

In [17]:
def hybrid_retrieve(query, k=3, alpha=0.5):
    vector_results = vectorstore.similarity_search_with_score(query, k=k)
    bm25_results = bm25_retrieve(query, k=k)

    combined_scores = {}

    for doc, score in vector_results:
        combined_scores[doc.page_content] = alpha * (1 / (1 + score))

    for score, doc in bm25_results:
        combined_scores[doc.page_content] = combined_scores.get(
            doc.page_content, 0
        ) + (1 - alpha) * score

    return sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)


Compare Vector vs Hybrid Results

In [18]:
query = "What is hybrid retrieval?"
print("VECTOR SEARCH RESULTS:\n")
vector_hits = vectorstore.similarity_search_with_score(query, k=3)

for doc, score in vector_hits:
    print(f"Score: {score:.4f}")
    print(doc.page_content)
    print()

VECTOR SEARCH RESULTS:

Score: 0.9470
Hybrid retrieval improves factual accuracy in RAG systems.

Score: 0.9831
Hybrid retrieval combines vector similarity and keyword matching.

Score: 1.3117
Vector search uses embeddings to retrieve semantically similar text.



In [19]:
query = "What is hybrid retrieval?"
print("HYBRID SEARCH RESULTS:\n")
hybrid_hits = hybrid_retrieve(query, k=3)

for text, score in hybrid_hits:
    print(f"Hybrid Score: {score:.4f}")
    print(text)
    print()

HYBRID SEARCH RESULTS:

Hybrid Score: 0.5380
FAISS is a library for efficient vector similarity search.

Hybrid Score: 0.2568
Hybrid retrieval improves factual accuracy in RAG systems.

Hybrid Score: 0.2521
Hybrid retrieval combines vector similarity and keyword matching.

Hybrid Score: 0.2163
Vector search uses embeddings to retrieve semantically similar text.

